In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
import catboost
from catboost import CatBoostRegressor
import lightgbm as lgb
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import AdaBoostRegressor
import optuna

In [2]:
df = pd.read_csv('전체병합_전처리.csv',index_col=0)

df_train=df[df['계약년도']!=2023].drop(columns=['구','계약년도','계약월','인구수'])
df_test=df[df['계약년도']==2023].drop(columns=['구','계약년도','계약월','인구수'])

X = df_train.drop(columns='평당가격')
y = df_train['평당가격']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=34)

X_test = df_test.drop(columns='평당가격')
y_test = df_test['평당가격']

Catboost
=

In [3]:
def catboost_objective(trial):
    # CatBoost 하이퍼파라미터 탐색 범위 설정
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 100),
    }
    
    # CatBoost 모델 생성
    model = CatBoostRegressor(**params, verbose=0)
    
    # 모델 훈련
    model.fit(X_train, y_train)
    
    # 검증 데이터에 대한 예측
    y_pred = model.predict(X_valid)
    
    # RMSE 및 MAE 계산
    rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
    mae = mean_absolute_error(y_valid, y_pred)
    
    # 최적화 목표 (RMSE) 반환
    return rmse

# Optuna 최적화 실행
study = optuna.create_study(direction='minimize')
study.optimize(catboost_objective, n_trials=100)

[I 2023-08-27 21:30:57,978] A new study created in memory with name: no-name-fdf6be75-73e0-4854-b2c0-a9dac25dabe7
[I 2023-08-27 21:31:01,891] Trial 0 finished with value: 393.1649319568944 and parameters: {'iterations': 797, 'learning_rate': 0.08331779958136774, 'depth': 4, 'l2_leaf_reg': 11.462184123573914}. Best is trial 0 with value: 393.1649319568944.
[I 2023-08-27 21:31:07,158] Trial 1 finished with value: 394.7465958451492 and parameters: {'iterations': 701, 'learning_rate': 0.032557746543287545, 'depth': 7, 'l2_leaf_reg': 55.40324897278397}. Best is trial 0 with value: 393.1649319568944.
[I 2023-08-27 21:31:25,373] Trial 2 finished with value: 706.8950712778978 and parameters: {'iterations': 899, 'learning_rate': 0.001630863152192844, 'depth': 10, 'l2_leaf_reg': 21.95109709958927}. Best is trial 0 with value: 393.1649319568944.
[I 2023-08-27 21:31:28,092] Trial 3 finished with value: 466.3339718365715 and parameters: {'iterations': 329, 'learning_rate': 0.03137282150591058, 'dep

In [6]:
# 최적 하이퍼파라미터 및 결과 출력
print("Best trial:")
best_trial = study.best_trial
print("  Value (RMSE): {}".format(best_trial.value))
print("  Params: ")
for key, value in best_trial.params.items():
    print("    {}: {}".format(key, value))

Best trial:
  Value (RMSE): 241.05407182065318
  Params: 
    iterations: 951
    learning_rate: 0.09080290708146298
    depth: 10
    l2_leaf_reg: 3.24879692712503


In [8]:
# CatBoost 모델 로드 또는 생성
best_params = best_trial.params
best_model = catboost.CatBoostRegressor(**best_params, verbose=0)

best_model.fit(X, y)
final_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, final_pred)
rmse = np.sqrt(mean_squared_error(y_test, final_pred))

In [9]:
# MAE와 RMSE 출력
print("MAE:", mae)
print("RMSE:", rmse)

MAE: 382.1243469610784
RMSE: 511.5748454108446


Adaboost
=

In [15]:
# Optuna objective 함수 정의
def objective(trial):
    # AdaBoost 모델 생성
    ada_boost_model = AdaBoostRegressor(
        n_estimators=trial.suggest_int('n_estimators', 50, 200),  # 추정기 개수
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.2),  # 학습률
        random_state=42  # 시드 설정 (선택 사항)
    )

    # 모델 훈련
    ada_boost_model.fit(X_train, y_train)

    # 검증 데이터로 예측
    y_pred = ada_boost_model.predict(X_valid)

    # RMSE 계산 및 반환 (Optuna는 최소화 문제를 풉니다)
    rmse = mean_squared_error(y_valid, y_pred, squared=False)
    return rmse

# Optuna 최적화 실행
study = optuna.create_study(direction='minimize')  # RMSE를 최소화하려고 합니다.
study.optimize(objective, n_trials=100)  # 시행 횟수를 조정하십시오.

[I 2023-08-27 21:57:41,064] A new study created in memory with name: no-name-fe4b2763-6bb1-46c8-a026-b16010a45e3e
[I 2023-08-27 21:58:02,187] Trial 0 finished with value: 723.3668324795557 and parameters: {'n_estimators': 73, 'learning_rate': 0.17327229745959705}. Best is trial 0 with value: 723.3668324795557.
[I 2023-08-27 21:58:19,775] Trial 1 finished with value: 697.9136879781282 and parameters: {'n_estimators': 55, 'learning_rate': 0.05929781146296601}. Best is trial 1 with value: 697.9136879781282.
[I 2023-08-27 21:58:43,311] Trial 2 finished with value: 702.2044558531711 and parameters: {'n_estimators': 74, 'learning_rate': 0.0355670256112299}. Best is trial 1 with value: 697.9136879781282.
[I 2023-08-27 21:59:17,432] Trial 3 finished with value: 738.1561199797301 and parameters: {'n_estimators': 117, 'learning_rate': 0.1535723268172283}. Best is trial 1 with value: 697.9136879781282.
[I 2023-08-27 22:00:05,442] Trial 4 finished with value: 718.6843169746022 and parameters: {'n_

In [16]:
# 최적 하이퍼파라미터 및 결과 출력
print("Best trial:")
best_trial = study.best_trial
print("  Value (RMSE): {}".format(best_trial.value))
print("  Params: ")
for key, value in best_trial.params.items():
    print("    {}: {}".format(key, value))

Best trial:
  Value (RMSE): 695.932067282677
  Params: 
    n_estimators: 78
    learning_rate: 0.047918330121882124


In [17]:
# 최적의 하이퍼파라미터로 모델 생성 및 평가
best_params = best_trial.params
best_model = AdaBoostRegressor(**best_params)
best_model.fit(X, y)
final = best_model.predict(X_test)
mae = mean_absolute_error(y_test, final)
rmse = np.sqrt(mean_squared_error(y_test, final))

In [18]:
# MAE와 RMSE 출력
print("MAE:", mae)
print("RMSE:", rmse)

MAE: 648.503941067455
RMSE: 841.3519411746566


Linear Regression
=

In [19]:
LRmodel = LinearRegression()
LRmodel.fit(X, y)

y_test_pred = LRmodel.predict(X)

# RMSE 및 MAE 계산 (검증 데이터에 대한 성능 평가)
rmse_test = np.sqrt(mean_squared_error(y, y_test_pred))
mae_test = mean_absolute_error(y, y_test_pred)

print(f'test RMSE: {rmse_test:.2f}')
print(f'test MAE: {mae_test:.2f}')

test RMSE: 1027.57
test MAE: 687.09
